# Workout — SQL + ORM SQLModel

## Objetivo

Practicar **SQLModel** (ORM sobre SQLite).

**ORM** = Object-Relational Mapper: hablamos con la base de datos usando **clases Python**
en lugar de escribir SQL a mano (aunque por debajo sí se genera SQL).

La demo va sobre tablas SQL que relacionan lugares (`Venue`) y eventos (`Event`) en Madrid. Veréis:

- `table=True` + `Field` — clase → tabla
- herencia (`CulturalItem` → `Event`)
- FK + `Relationship` + join — relacionar tablas
- `Session`: add / commit / select / update / delete
- validación y `try/except`

## Índice

1. Setup
2. Modelos
3. Engine y tablas
4. Insertar datos de demo
5. Listar y join
6. Buscar y actualizar
7. Crear / borrar
8. Validación que falla
9. Cierre

## 1. Setup

Este workout es **solo este notebook**. La primera vez, instalad SQLModel con la celda de abajo
e importad lo mínimo.

| Import | Para qué |
|--------|----------|
| `SQLModel`, `Field` | Definir modelos/tablas |
| `create_engine` | Conectar con SQLite |
| `Session` | Abrir una "sesión" de trabajo con la DB |
| `select` | Consultas (SELECT …) |
| `Relationship` | Enlazar objetos en Python (venue ↔ events) |

In [ ]:
# Primera vez (o si falta el paquete en el kernel):
%pip install -qU sqlmodel

In [1]:
import os
from typing import Optional  # Optional[X] = puede ser X o None

from pydantic import ValidationError, field_validator
from sqlmodel import (
    Field,          # reglas de columna (PK, FK, longitudes…)
    Relationship,   # relación entre modelos en Python
    Session,        # unidad de trabajo con la DB
    SQLModel,       # base de los modelos
    create_engine,  # conexión (aquí: fichero SQLite)
    select,         # construir consultas SELECT
)

## 2. Modelos

En SQLModel, una **clase** con `table=True` = una **tabla** SQL.
Los atributos = columnas. Usamos OOP (clases) para declarar el esquema.

| Clase | Rol |
|-------|-----|
| `Venue` | Lugar (tabla) |
| `CulturalItem` | Base compartida (**no** es tabla: sin `table=True`) |
| `Event` | Evento (tabla) + FK a `Venue` |

### Relación 1–N (uno a muchos)

```text
Venue (1) ──────── < Event (N)
  un lugar              muchos eventos
```

- **Un** `Venue` puede tener **muchos** `Event`.
- **Cada** `Event` pertenece a **un** `Venue` (vía `venue_id`).

Ejemplo: Museo Reina Sofía (1 venue) → «Colección permanente» + «Exposición temporal» (N eventos).

**En SQL (base de datos):** la Foreign Key (FK).

- **`foreign_key="venue.id"`** en `Event.venue_id` → columna real en la tabla que apunta a `venue.id`.
- Eso es lo que **une las tablas** en SQLite y define el 1–N.
- El join de más abajo también usa la FK (`Event.venue_id == Venue.id`).

**En Python (SQLModel):** `Relationship` y `back_populates`.

- **`Relationship`** es un atajo: podéis escribir `event.venue` (el lugar de ese evento)
  o `venue.events` (lista de eventos del lugar) sin montar el join a mano cada vez.
- **`back_populates`** indica el nombre del atributo **en la otra clase**, para que los dos extremos
  queden enlazados en el código:

  ```text
  En Event:   venue  = Relationship(back_populates="events")
                              ↑
                    "en Venue, el atributo se llama events"

  En Venue:   events = Relationship(back_populates="venue")
                              ↑
                    "en Event, el atributo se llama venue"
  ```

Resumen: la FK une tablas en SQL; `Relationship` + `back_populates` permiten navegar la misma
relación desde Python en ambos sentidos (`event.venue` ↔ `venue.events`).

In [2]:
class Venue(SQLModel, table=True):
    """Lugar cultural (museo, sala, teatro…). table=True → crea tabla 'venue'."""

    # primary_key=True → clave primaria (identifica la fila)
    # default=None → SQLite asigna el id al hacer commit
    id: Optional[int] = Field(default=None, primary_key=True)

    # ... = obligatorio; min_length evita strings vacíos
    name: str = Field(..., min_length=1, description="Nombre del lugar")
    barrio: str = Field(..., min_length=1, description="Barrio o zona")

    # En Python: lista de events de este venue (emparejar con Event.venue)
    # back_populates = "el otro extremo se llama venue"; lo importante en SQL es la FK
    events: list["Event"] = Relationship(back_populates="venue")

    @field_validator("name", "barrio", mode="before")
    @classmethod
    def no_vacio(cls, value: str) -> str:
        # Validación extra: quitar espacios; rechazar solo espacios
        if not str(value).strip():
            raise ValueError("No puede estar vacío.")
        return str(value).strip()


class CulturalItem(SQLModel):
    """Base compartida. Sin table=True → NO crea tabla propia."""

    id: Optional[int] = Field(default=None, primary_key=True)
    title: str = Field(..., min_length=1)
    description: Optional[str] = Field(default=None, max_length=500)

    @field_validator("title", mode="before")
    @classmethod
    def title_ok(cls, value: str) -> str:
        if not str(value).strip():
            raise ValueError("El título no puede estar vacío.")
        return str(value).strip()


class Event(CulturalItem, table=True):
    """Evento = hereda title/description de CulturalItem + FK a Venue."""

    when_label: str = Field(default="por confirmar", max_length=100)

    # FK (lo importante): columna SQL que apunta a venue.id → relación 1–N
    venue_id: Optional[int] = Field(default=None, foreign_key="venue.id")

    # En Python: event.venue; back_populates="events" empareja con Venue.events
    venue: Optional[Venue] = Relationship(back_populates="events")


print("Modelos definidos:", Venue.__name__, Event.__name__)

Modelos definidos: Venue Event


## 3. Engine y tablas

**SQLite** guarda la base de datos en un fichero (`cultural_demo.db`) en esta carpeta.

- `create_engine` → abrir conexión.
- `create_all` → crear tablas según los modelos.
- `drop_all` aquí es solo para el notebook: **reset** limpio si re-ejecutáis las celdas.

> Si no hicierais reset y volvierais a insertar, duplicaríais filas.

**Ver el fichero `.db` en el editor**

Tras ejecutar la celda de abajo, aparece `cultural_demo.db` en esta carpeta.
En VS Code / Cursor podéis abrirlo con una extensión de SQLite (por ejemplo *SQLite Viewer*):
clic derecho en el fichero → **Open With** → la extensión.
Así veis las tablas `venue` y `event` sin salir del IDE.

In [3]:
DB_FILE = "cultural_demo.db"

# sqlite:///nombre.db → fichero local (no hace falta servidor)
# echo=False → no imprime el SQL generado (poned True si queréis verlo)
engine = create_engine(f"sqlite:///{DB_FILE}", echo=False)

# Reset del notebook: borrar tablas antiguas y crearlas de nuevo
SQLModel.metadata.drop_all(engine)    # DROP TABLE …
SQLModel.metadata.create_all(engine)  # CREATE TABLE … (según Venue/Event)

print(f"Engine listo → {DB_FILE}")
print("Tablas recreadas.")

Engine listo → cultural_demo.db
Tablas recreadas.


## 4. Insertar datos de demo

Patrón de escritura:

1. Abrir `Session`
2. Crear objetos Python (`Venue`, `Event`)
3. `add` / `add_all` — preparar el insert
4. `commit` — **guardar de verdad** en el `.db`

Si pasáis `venue=reina` al crear un `Event`, SQLModel rellena `venue_id` solo.

In [5]:
# with Session → abre y cierra la sesión (buena práctica)
with Session(engine) as session:
    # Objetos en memoria (aún no están en el .db hasta el commit)
    reina = Venue(name="Museo Reina Sofía", barrio="Lavapiés")
    prado = Venue(name="Museo del Prado", barrio="Retiro")
    sala = Venue(name="Sala Villanos", barrio="Malasaña")

    events = [
        Event(
            title="Colección permanente",
            description="Arte del siglo XX",
            when_label="todos los días",
            venue=reina,  # SQLModel pone venue_id al guardar
        ),
        Event(
            title="Exposición temporal",
            description="Muestra rotativa",
            when_label="este mes",
            venue=reina,
        ),
        Event(
            title="Visita guiada breve",
            description="Recorrido 45 min",
            when_label="sábados mañana",
            venue=prado,
        ),
        Event(
            title="Sesión de jazz",
            description="Entrada asequible",
            when_label="viernes noche",
            venue=sala,
        ),
    ]

    # add_all = marcar para INSERT; commit = ejecutar y persistir
    session.add_all([reina, prado, sala] + events)
    session.commit()

print("Datos de demo insertados.")

Datos de demo insertados.


## 5. Listar y join

Lectura = `select(...)` + `session.exec(...).all()`.

1. Solo venues: `select(Venue)` ≈ `SELECT * FROM venue`.
2. **Join**: juntar `Event` y `Venue` por la FK (`event.venue_id == venue.id`).
   Así obtenéis el nombre del lugar junto al evento (INNER JOIN).

In [6]:
with Session(engine) as session:
    # select(Venue) → todas las filas de la tabla venue
    # .all() → lista de objetos Venue
    venues = session.exec(select(Venue)).all()

print("Venues:")
for v in venues:
    print(f"  - [{v.id}] {v.name} ({v.barrio})")

Venues:
  - [1] Museo Reina Sofía (Lavapiés)
  - [2] Museo del Prado (Retiro)
  - [3] Sala Villanos (Malasaña)
  - [4] Museo Reina Sofía (Lavapiés)
  - [5] Museo del Prado (Retiro)
  - [6] Sala Villanos (Malasaña)


In [7]:
with Session(engine) as session:
    # Join: cada fila del resultado es (Event, Venue)
    # Condición: la FK del evento coincide con el id del venue
    rows = session.exec(
        select(Event, Venue).join(Venue, Event.venue_id == Venue.id)
    ).all()

print("Eventos con su venue:")
for event, venue in rows:
    print(
        f"  - {event.title} @ {venue.name} "
        f"({venue.barrio}) — {event.when_label}"
    )

Eventos con su venue:
  - Colección permanente @ Museo Reina Sofía (Lavapiés) — todos los días
  - Exposición temporal @ Museo Reina Sofía (Lavapiés) — este mes
  - Visita guiada breve @ Museo del Prado (Retiro) — sábados mañana
  - Sesión de jazz @ Sala Villanos (Malasaña) — viernes noche
  - Colección permanente @ Museo Reina Sofía (Lavapiés) — todos los días
  - Exposición temporal @ Museo Reina Sofía (Lavapiés) — este mes
  - Visita guiada breve @ Museo del Prado (Retiro) — sábados mañana
  - Sesión de jazz @ Sala Villanos (Malasaña) — viernes noche


## 6. Buscar y actualizar

1. `where(...)` filtra (≈ `WHERE title = ...`).
2. `.first()` → una fila o `None`.
3. Cambiáis el atributo en Python → `add` + `commit` = **UPDATE** en SQL.

In [8]:
with Session(engine) as session:
    # Buscar un evento por título
    event = session.exec(
        select(Event).where(Event.title == "Sesión de jazz")
    ).first()

    if event:
        print(f"Antes: {event.title} → {event.when_label}")
        event.when_label = "sábado 21:00"  # cambio en memoria
        session.add(event)                 # marcar para UPDATE
        session.commit()                   # persistir
        print(f"Después: {event.title} → {event.when_label}")
    else:
        print("No se encontró «Sesión de jazz».")

Antes: Sesión de jazz → viernes noche
Después: Sesión de jazz → sábado 21:00


In [9]:
# Comprobar que el UPDATE se ve en un join de nuevo
with Session(engine) as session:
    rows = session.exec(
        select(Event, Venue).join(Venue, Event.venue_id == Venue.id)
    ).all()

for event, venue in rows:
    print(f"  - {event.title} @ {venue.name} — {event.when_label}")

  - Colección permanente @ Museo Reina Sofía — todos los días
  - Exposición temporal @ Museo Reina Sofía — este mes
  - Visita guiada breve @ Museo del Prado — sábados mañana
  - Sesión de jazz @ Sala Villanos — sábado 21:00
  - Colección permanente @ Museo Reina Sofía — todos los días
  - Exposición temporal @ Museo Reina Sofía — este mes
  - Visita guiada breve @ Museo del Prado — sábados mañana
  - Sesión de jazz @ Sala Villanos — viernes noche


## 7. Crear / borrar

- **Crear:** validar datos → `add` → `commit` (otro INSERT).
- **`refresh`:** recargar el objeto desde la DB (p. ej. para tener el `id` asignado).
- **Borrar:** `session.delete(obj)` + `commit` (DELETE).

In [ ]:
# Crear venue + evento (con validación previa)
try:
    # model_validate: comprueba Field/validators antes de tocar la DB
    validated = Venue.model_validate({"name": "Matadero", "barrio": "Arganzuela"})

    with Session(engine) as session:
        venue = Venue(**validated.model_dump())
        session.add(venue)
        session.commit()
        session.refresh(venue)  # venue.id ya viene de SQLite

        ev = Event(
            title="Mercado de diseño",
            description="Feria de fin de semana",
            when_label="fin de semana",
            venue=venue,
        )
        session.add(ev)
        session.commit()

    print("Creados Matadero + Mercado de diseño.")
except ValidationError as err:
    print("Error de validación:")
    print(err.json(indent=2))

In [ ]:
# Borrar un evento por título
with Session(engine) as session:
    event = session.exec(
        select(Event).where(Event.title == "Exposición temporal")
    ).first()

    if event:
        session.delete(event)  # marcar DELETE
        session.commit()       # aplicar
        print("Eliminado: Exposición temporal")
    else:
        print("No había «Exposición temporal» (¿ya lo borraste?).")

## 8. Validación que falla

SQLModel usa Pydantic: datos inválidos → `ValidationError` **antes** de escribir en la DB.
Aquí un título vacío debe fallar (`min_length` / validator).

In [ ]:
try:
    # Título vacío → no pasa la validación del modelo
    Event.model_validate({"title": "", "when_label": "hoy"})
    print("¿Sin error? No debería pasar.")
except ValidationError as err:
    print("ValidationError (esperado):")
    print(err)

## 9. Cierre

Conceptos de **SQLModel** que habéis tocado en este notebook:

| Concepto | Qué hace |
|----------|----------|
| `SQLModel` + `table=True` | Una clase Python = una tabla en la DB |
| `Field` | Columnas, tipos, defaults, validación |
| FK (`foreign_key`) | Une tablas en SQL (p. ej. `Event.venue_id` → `Venue.id`) |
| `Relationship` + `back_populates` | Navegar la misma relación en Python (`event.venue` ↔ `venue.events`) |
| Herencia (`CulturalItem` → `Event`) | Campos compartidos sin duplicar; solo `Event` es tabla |
| `create_engine` + SQLite | Conexión a un fichero `.db` (sin servidor) |
| `Session` + `add` / `commit` | Escribir en la BBDD (INSERT / UPDATE / DELETE) |
| `select` / `where` / `join` | Leer y relacionar filas |
| `ValidationError` (Pydantic) | Rechazar datos inválidos **antes** de llegar a la DB |

### Checklist

- [ ] Distingo FK (SQL) de `Relationship` / `back_populates` en SQLModel
- [ ] Sé qué implica `table=True` y por qué `CulturalItem` no lo lleva
- [ ] Entiendo el ciclo `Session` → `add` → `commit` (y qué pasa sin `commit`)
- [ ] Puedo montar un `select` con `where` y un `join` por FK
- [ ] Sé que SQLModel valida con Pydantic (`ValidationError`) antes de persistir